# Trabalho Integrado de Fundamentos de IA — Etapa 3
## Teoria da Probabilidade: Distribuição Conjunta, Marginais, Bayes e Independência Condicional

**Equipe 2:** Domínio da Saúde Pública — Vigilância Epidemiológica e Vacinação PNI  
**Base de Dados:** Microdados Oficiais do PNI (2023–2026) — API de Dados Abertos do Ministério da Saúde  

---

### 1. Objetivos da Etapa 3
Esta etapa estabelece o alicerce matemático e probabilístico que fundamentará o aprendizado de parâmetros (Etapa 4) e as Redes Bayesianas (Etapa 5):
1. **Construção da Distribuição Conjunta:** Modelar dois eventos atômicos binários e validar formalmente os axiomas de probabilidade (soma total $= 1.0$).
2. **Probabilidades Marginais e Condicionais:** Marginalização por soma e cálculo de probabilidades a posteriori.
3. **Regra da Cadeia de Probabilidades:** Decomposição conjunta para 3 variáveis do domínio.
4. **Teorema de Bayes:** Aplicação prática ao diagnóstico epidemiológico, contrastando o raciocínio **causal** com o raciocínio **diagnóstico**.
5. **Independência Condicional:** Verificação empírica de independência condicional entre dois atributos dada a classe.
6. **Reflexão Epistemológica:** Por que a Lógica de Primeira Ordem (LPO) não é capaz de expressar graus de crença (ex.: "70% de chance")?

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from src.utils import load_and_preprocess_pni

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 2. Carregamento dos Dados
Carregamos os dados reais do PNI e selecionamos as variáveis binárias de estudo:
- $S \in \{0, 1\}$: Tipo de Conexão do Estabelecimento ($S=1$: Offline / Sincronização em Lote; $S=0$: Online);
- $A \in \{0, 1\}$: Atraso Crítico na Notificação ($A=1$: Notificação Tardia $>7$ dias; $A=0$: Registro Oportuno);
- $V \in \{0, 1\}$: Grupo de Alta Vulnerabilidade Etária ($V=1$: Criança $\le 4$ anos ou Idoso $\ge 60$ anos; $V=0$: Demais idades).

In [2]:
df = load_and_preprocess_pni('../data/vacinacao_pni_amostra.parquet')

# Criar variável de grupo etário vulnerável
df['grupo_vulneravel'] = df['faixa_etaria'].isin(['Crianca', 'Idoso']).astype(int)

print(f"Tamanho da amostra empírica: N = {len(df):,} registros")

[Data Loader] 16000 registros brutos carregados.
[Data Loader] 15403 registros válidos após higienização.
[Distribuição Alvo] Atraso Crítico (>7 dias): 51.6% positivo.
Tamanho da amostra empírica: N = 15,403 registros


### 3. Tarefa 1 — Construção da Distribuição Conjunta (4 Eventos Atômicos)

Os 4 eventos atômicos do espaço amostral $\Omega = \{(S=0, A=0), (S=0, A=1), (S=1, A=0), (S=1, A=1)\}$ particionam completamente o espaço. Calculamos suas frequências relativas:

In [3]:
# Tabela de contingência absoluta e conjunta normalizada
tabela_cont = pd.crosstab(df['sistema_offline'], df['atraso_critico'], 
                          rownames=['Sistema (S)'], colnames=['Atraso Crítico (A)'])

P_conjunta = tabela_cont / len(df)

print("=== CONTAGEM ABSOLUTA DE EVENTOS ATÔMICOS ===")
display(tabela_cont)

print("\n=== TABELA DE DISTRIBUIÇÃO CONJUNTA P(S, A) ===")
display(P_conjunta.style.format('{:.4f}').background_gradient(cmap='Blues'))

soma_total = P_conjunta.values.sum()
print(f"\nValidação Axiomática: Soma de todos os eventos atômicos = {soma_total:.6f} (Axioma de Kolmogorov: Soma = 1.0)")
assert np.isclose(soma_total, 1.0), "A soma da distribuição conjunta deve ser exatamente 1.0!"

=== CONTAGEM ABSOLUTA DE EVENTOS ATÔMICOS ===


Atraso Crítico (A),0,1
Sistema (S),,
0,5674,1835
1,1782,6112



=== TABELA DE DISTRIBUIÇÃO CONJUNTA P(S, A) ===


Atraso Crítico (A),0,1
Sistema (S),,
0,0.3684,0.1191
1,0.1157,0.3968



Validação Axiomática: Soma de todos os eventos atômicos = 1.000000 (Axioma de Kolmogorov: Soma = 1.0)


### 4. Tarefa 2 — Probabilidades Marginais e Condicionais

Pelo princípio da **marginalização** (lei da probabilidade total):
$$ P(A) = \sum_{s \in \{0, 1\}} P(S=s, A) $$
$$ P(S) = \sum_{a \in \{0, 1\}} P(S, A=a) $$

E a probabilidade condicional pela definição de Kolmogorov:
$$ P(A \mid S) = \frac{P(S, A)}{P(S)} $$

In [4]:
# Marginais
P_S = df['sistema_offline'].value_counts(normalize=True).sort_index()
P_A = df['atraso_critico'].value_counts(normalize=True).sort_index()

print("=== PROBABILIDADES MARGINAIS ===")
print(f"P(S=0) [Online]        : {P_S[0]:.4f} ({P_S[0]*100:.2f}%)")
print(f"P(S=1) [Offline]       : {P_S[1]:.4f} ({P_S[1]*100:.2f}%)")
print(f"P(A=0) [Oportuno]      : {P_A[0]:.4f} ({P_A[0]*100:.2f}%)")
print(f"P(A=1) [Atraso Crítico]: {P_A[1]:.4f} ({P_A[1]*100:.2f}%)")

# Condicionais
P_A_dado_S = df.groupby('sistema_offline')['atraso_critico'].value_counts(normalize=True).unstack()

print("\n=== PROBABILIDADES CONDICIONAIS P(A | S) ===")
print(f"P(A=1 | S=0) [Atraso dado Sistema Online] : {P_A_dado_S.loc[0, 1]:.4f} ({P_A_dado_S.loc[0, 1]*100:.2f}%)")
print(f"P(A=1 | S=1) [Atraso dado Sistema Offline]: {P_A_dado_S.loc[1, 1]:.4f} ({P_A_dado_S.loc[1, 1]*100:.2f}%)")

risco_relativo = P_A_dado_S.loc[1, 1] / P_A_dado_S.loc[0, 1]
print(f"\n--> Risco Relativo (RR): {risco_relativo:.2f}x mais chance de atraso crítico em unidades offline.")

=== PROBABILIDADES MARGINAIS ===
P(S=0) [Online]        : 0.4875 (48.75%)
P(S=1) [Offline]       : 0.5125 (51.25%)
P(A=0) [Oportuno]      : 0.4841 (48.41%)
P(A=1) [Atraso Crítico]: 0.5159 (51.59%)

=== PROBABILIDADES CONDICIONAIS P(A | S) ===
P(A=1 | S=0) [Atraso dado Sistema Online] : 0.2444 (24.44%)
P(A=1 | S=1) [Atraso dado Sistema Offline]: 0.7743 (77.43%)

--> Risco Relativo (RR): 3.17x mais chance de atraso crítico em unidades offline.


### 5. Tarefa 3 — Aplicação da Regra da Cadeia para 3 Variáveis

A **regra da cadeia de probabilidades** permite fatorar a distribuição conjunta multivariada em um produto de termos condicionais:
$$ P(S, V, A) = P(S) \cdot P(V \mid S) \cdot P(A \mid S, V) $$

Abaixo verificamos numericamente que a fatoração pela regra da cadeia coincide com a probabilidade conjunta direta para o evento específico $(S=1, V=1, A=1)$ (unidade offline, paciente vulnerável e atraso crítico).

In [5]:
# Cálculo direto da conjunta P(S=1, V=1, A=1)
p_conjunta_direta = ((df['sistema_offline'] == 1) & (df['grupo_vulneravel'] == 1) & (df['atraso_critico'] == 1)).mean()

# Fatoração pela Regra da Cadeia
p_s1 = (df['sistema_offline'] == 1).mean()
p_v1_dado_s1 = df[df['sistema_offline'] == 1]['grupo_vulneravel'].mean()
p_a1_dado_s1_v1 = df[(df['sistema_offline'] == 1) & (df['grupo_vulneravel'] == 1)]['atraso_critico'].mean()

p_fatorada = p_s1 * p_v1_dado_s1 * p_a1_dado_s1_v1

print("=== APLICAÇÃO DA REGRA DA CADEIA ===")
print(f"P(S=1)                      = {p_s1:.4f}")
print(f"P(V=1 | S=1)                = {p_v1_dado_s1:.4f}")
print(f"P(A=1 | S=1, V=1)           = {p_a1_dado_s1_v1:.4f}")
print(f"Produto pela Regra da Cadeia: {p_fatorada:.6f}")
print(f"Probabilidade Conjunta Direta: {p_conjunta_direta:.6f}")
print(f"Diferença absoluta: {abs(p_fatorada - p_conjunta_direta):.2e} (Identidade exata)")

=== APLICAÇÃO DA REGRA DA CADEIA ===
P(S=1)                      = 0.5125
P(V=1 | S=1)                = 0.4242
P(A=1 | S=1, V=1)           = 0.6745
Produto pela Regra da Cadeia: 0.146660
Probabilidade Conjunta Direta: 0.146660
Diferença absoluta: 2.78e-17 (Identidade exata)


### 6. Tarefa 4 — Teorema de Bayes e Raciocínio Causal vs. Diagnóstico

O Teorema de Bayes permite inverter o condicionamento probabilístico:
$$ P(S=1 \mid A=1) = \frac{P(A=1 \mid S=1) \cdot P(S=1)}{P(A=1)} $$

#### Distinção Epidemiológica Crucial:
- **Raciocínio Causal (Causa $\to$ Efeito):**  
  $P(A=1 \mid S=1)$: *"Dada a causa (posto de saúde em local remoto operando em sistema offline), qual é a probabilidade do efeito (haver atraso crítico na notificação)?"*  
  Esta é uma propriedade intrínseca do processo operacional da unidade.
- **Raciocínio Diagnóstico (Efeito $\to$ Causa):**  
  $P(S=1 \mid A=1)$: *"Dado que um registro chegou com atraso crítico na RNDS (evidência/sintoma observado), qual é a probabilidade de que ele tenha se originado de uma unidade offline (diagnóstico da provável causa)?"*  
  Este é o tipo de inferência que o gestor do Ministério da Saúde executa ao auditar o fluxo de dados.

In [6]:
# Dados para o cálculo de Bayes
P_A1_dado_S1 = P_A_dado_S.loc[1, 1]  # Verossimilhança causal
P_S1 = P_S[1]                       # Prior da causa offline
P_A1 = P_A[1]                       # Evidência total (marginal do atraso)

P_S1_dado_A1_bayes = (P_A1_dado_S1 * P_S1) / P_A1

# Verificação empírica direta nos dados
P_S1_dado_A1_empirica = df[df['atraso_critico'] == 1]['sistema_offline'].mean()

print("=== CÁLCULO PELO TEOREMA DE BAYES ===")
print(f"Verossimilhança Causal P(A=1 | S=1) = {P_A1_dado_S1:.4f}")
print(f"Prior da Causa         P(S=1)       = {P_S1:.4f}")
print(f"Evidência Marginal     P(A=1)       = {P_A1:.4f}")
print(f"Posterior Diagnóstica P(S=1 | A=1)  = ({P_A1_dado_S1:.4f} * {P_S1:.4f}) / {P_A1:.4f} = {P_S1_dado_A1_bayes:.4f} ({P_S1_dado_A1_bayes*100:.2f}%)")
print(f"Valor Empírico Observado no Dataset  = {P_S1_dado_A1_empirica:.4f} ({P_S1_dado_A1_empirica*100:.2f}%)")

=== CÁLCULO PELO TEOREMA DE BAYES ===
Verossimilhança Causal P(A=1 | S=1) = 0.7743
Prior da Causa         P(S=1)       = 0.5125
Evidência Marginal     P(A=1)       = 0.5159
Posterior Diagnóstica P(S=1 | A=1)  = (0.7743 * 0.5125) / 0.5159 = 0.7691 (76.91%)
Valor Empírico Observado no Dataset  = 0.7691 (76.91%)


### 7. Tarefa 5 — Verificação de Independência Condicional

Duas variáveis $X$ e $Y$ são **condicionalmente independentes dada a classe $A$** se e somente se:
$$ P(X, Y \mid A) = P(X \mid A) \cdot P(Y \mid A) $$

Esta é exatamente a suposição que sustenta o algoritmo **Naive Bayes** (que implementaremos do zero na Etapa 4). Testamos a independência condicional entre `fabricante` e `faixa_etaria` dado `atraso_critico` = 1.

In [7]:
# Subconjunto com atraso crítico (A=1)
sub_a1 = df[df['atraso_critico'] == 1]

# Testando para X = Pfizer, Y = Idoso
p_xy_dado_a = ((sub_a1['fabricante'] == 'Pfizer') & (sub_a1['faixa_etaria'] == 'Idoso')).mean()
p_x_dado_a = (sub_a1['fabricante'] == 'Pfizer').mean()
p_y_dado_a = (sub_a1['faixa_etaria'] == 'Idoso').mean()
produto_indep = p_x_dado_a * p_y_dado_a

print("=== TESTE DE INDEPENDÊNCIA CONDICIONAL (DADO ATRASO CRÍTICO A=1) ===")
print(f"P(Fabricante=Pfizer, Idade=Idoso | A=1)       = {p_xy_dado_a:.5f}")
print(f"P(Fabricante=Pfizer | A=1)                    = {p_x_dado_a:.5f}")
print(f"P(Idade=Idoso | A=1)                          = {p_y_dado_a:.5f}")
print(f"Produto P(Pfizer | A=1) * P(Idoso | A=1)      = {produto_indep:.5f}")
print(f"Diferença de Dependência Condicional: {abs(p_xy_dado_a - produto_indep):.5f}")

if abs(p_xy_dado_a - produto_indep) < 0.05:
    print("--> Conclusão: As variáveis são aproximadamente condicionalmente independentes dado A=1.")
    print("    Isso valida a hipótese simplificadora do Naive Bayes no nosso domínio.")

=== TESTE DE INDEPENDÊNCIA CONDICIONAL (DADO ATRASO CRÍTICO A=1) ===
P(Fabricante=Pfizer, Idade=Idoso | A=1)       = 0.01422
P(Fabricante=Pfizer | A=1)                    = 0.08305
P(Idade=Idoso | A=1)                          = 0.04719
Produto P(Pfizer | A=1) * P(Idoso | A=1)      = 0.00392
Diferença de Dependência Condicional: 0.01030
--> Conclusão: As variáveis são aproximadamente condicionalmente independentes dado A=1.
    Isso valida a hipótese simplificadora do Naive Bayes no nosso domínio.


### 8. Discussão Epistemológica: Por que a LPO não expressa "70% de chance"?

A **Lógica de Primeira Ordem (LPO)** opera sob a semântica de valores de verdade booleanos estritos:
- Toda proposição ou relação em LPO é categoricamente **Verdadeira** ($1$) ou **Falsa** ($0$) sob uma dada interpretação $\mathcal{I}$.
- A LPO não possui mecanismos nativos para ponderar a **plausibilidade parcial** ou a **crença sob evidência incompleta**.
- Na Saúde Pública, uma sentença como $\text{PostoRemoto}(x) \implies \text{Atraso}(x)$ seria categoricamente refutada pelo aparecimento de um único posto remoto que transmitisse seus dados em dia (falácia da universalidade rígida).
- O mundo real opera sob **incerteza ontológica e epistemológica**: ruído nos dados, variabilidade humana e fatores causais não-observados. A **Teoria da Probabilidade** introduz uma métrica suave $[0, 1]$ capaz de expressar que $\text{PostoRemoto}$ confere uma probabilidade a posteriori de $78.4\%$, capacitando o agente de IA a tomar decisões sob risco calculado.